In [72]:
# Импорты
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

RS = 42

In [73]:
# Функция для вывода метрик

def print_cv_metrics(model_name, scores):
    print(f"\nМетрики для лучшей модели {model_name}:")
    for metric_name in ['accuracy', 'recall', 'precision', 'f1']:
        mean_score = scores[f'test_{metric_name}'].mean()
        std_score = scores[f'test_{metric_name}'].std()
        print(f"{metric_name.capitalize()} (CV): {mean_score:.4f} (+/- {std_score:.4f})")

In [74]:
# Функция для выбора лучшей модели GridSearchCV и RandomizedSearchCV

def evaluate_tuned_model(search_object, X_train, y_train, model_display_name):

    print(f"\nЛучшие параметры для {model_display_name}: {search_object.best_params_}")
    print(f"Лучшая оценка кросс-валидации для {model_display_name}: {search_object.best_score_:.4f}")

    best_model = search_object.best_estimator_
    if hasattr(best_model, 'named_steps') and 'classifier' in best_model.named_steps:
        print(f"Лучшая общая модель: {best_model.named_steps['classifier']}")
    else:
        print(f"Лучшая общая модель: {best_model}")

    scores = cross_validate(best_model, X_train, y_train, cv=10, scoring=['accuracy', 'recall', 'precision', 'f1'])
    print_cv_metrics(model_display_name, scores)
    return best_model

In [75]:
# Загрузка датасета

data = pd.read_csv('heart.csv')
data.info()
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [76]:
# Подготовка данных

X = data.drop('HeartDisease', axis=1)
y = data['HeartDisease']

# Категориальные и числовые признаки
categorical_cols = X.select_dtypes(include='object').columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
encoded_categorical_data = encoder.fit_transform(X[categorical_cols])
encoded_categorical_df = pd.DataFrame(encoded_categorical_data, columns=encoder.get_feature_names_out(categorical_cols), index=X.index)

# Масштабирование числовых признаков
scaler = StandardScaler()
X_scaled_numerical = scaler.fit_transform(X[numerical_cols])
X_scaled_numerical_df = pd.DataFrame(X_scaled_numerical, columns=numerical_cols, index=X.index)

X_scaled = pd.concat([X_scaled_numerical_df, encoded_categorical_df], axis=1)


print(X_scaled.head())

        Age  RestingBP  Cholesterol  FastingBS     MaxHR   Oldpeak  Sex_M  \
0 -1.433140   0.410909     0.825070  -0.551341  1.382928 -0.832432    1.0   
1 -0.478484   1.491752    -0.171961  -0.551341  0.754157  0.105664    0.0   
2 -1.751359  -0.129513     0.770188  -0.551341 -1.525138 -0.832432    1.0   
3 -0.584556   0.302825     0.139040  -0.551341 -1.132156  0.574711    0.0   
4  0.051881   0.951331    -0.034755  -0.551341 -0.581981 -0.832432    1.0   

   ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  \
0                1.0                0.0               0.0                1.0   
1                0.0                1.0               0.0                1.0   
2                1.0                0.0               0.0                0.0   
3                0.0                0.0               0.0                1.0   
4                0.0                1.0               0.0                1.0   

   RestingECG_ST  ExerciseAngina_Y  ST_Slope_Flat  ST_Sl

In [77]:
# Разбивка выборки на обучающую и тестовую

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=RS)

In [78]:
# Модель логистической регрессии

lr_model = LogisticRegression(random_state=RS)

scoring = ['accuracy', 'recall', 'precision', 'f1']
scores = cross_validate(lr_model, X_scaled, y, cv=10, scoring=scoring)

# Вывод результатов кросс-валидации
print_cv_metrics(lr_model, scores)


Метрики для лучшей модели LogisticRegression(random_state=42):
Accuracy (CV): 0.8506 (+/- 0.0582)
Recall (CV): 0.8695 (+/- 0.0992)
Precision (CV): 0.8667 (+/- 0.0637)
F1 (CV): 0.8639 (+/- 0.0571)


In [69]:
# Оптимизация параметров с помощью GridSearchCV

params = [
    {
        'solver': ['liblinear'],
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'max_iter': [100, 500, 1000]
    },
    {
        'solver': ['lbfgs', 'newton-cg', 'sag'],
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'], # Эти солверы поддерживают только L2
        'max_iter': [100, 500, 1000]
    },
    {
        'solver': ['saga'],
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'max_iter': [100, 500, 1000]
    }
]

grid = GridSearchCV(lr_model, param_grid=params, cv=10,)
grid.fit(X_train, y_train)
best_grid_model = evaluate_tuned_model(grid, X_train, y_train, "GridSearchCV")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 


Лучшие параметры для GridSearchCV: {'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Лучшая оценка кросс-валидации для GridSearchCV: 0.8677
Лучшая общая модель: LogisticRegression(C=1, random_state=42)

Метрики для лучшей модели GridSearchCV:
Accuracy (CV): 0.8677 (+/- 0.0360)
Recall (CV): 0.8927 (+/- 0.0318)
Precision (CV): 0.8708 (+/- 0.0444)
F1 (CV): 0.8810 (+/- 0.0307)


In [70]:
# Оптимизация параметров с помощью RandomizedSearchCV

r_search = RandomizedSearchCV(lr_model, param_distributions=params, random_state=RS)
r_search.fit(X_train, y_train)
best_r_search_model = evaluate_tuned_model(r_search, X_train, y_train, "RandomizedSearchCV (только LogisticRegression)")


Лучшие параметры для RandomizedSearchCV (только LogisticRegression): {'solver': 'saga', 'penalty': 'l2', 'max_iter': 100, 'C': 1}
Лучшая оценка кросс-валидации для RandomizedSearchCV (только LogisticRegression): 0.8665
Лучшая общая модель: LogisticRegression(C=1, random_state=42, solver='saga')

Метрики для лучшей модели RandomizedSearchCV (только LogisticRegression):
Accuracy (CV): 0.8677 (+/- 0.0360)
Recall (CV): 0.8927 (+/- 0.0318)
Precision (CV): 0.8708 (+/- 0.0444)
F1 (CV): 0.8810 (+/- 0.0307)


In [71]:
# Дополнительные модели и их параметры для RandomizedSearchCV
params_extended = [
    {
        'classifier': [LogisticRegression(random_state=RS)],
        'classifier__solver' : ['liblinear'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__max_iter': [100, 500, 1000]
    },
    {
        'classifier': [LogisticRegression(random_state=RS)],
        'classifier__solver': ['lbfgs', 'newton-cg', 'sag'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l2'], # Эти солверы поддерживают только L2
        'classifier__max_iter': [100, 500, 1000]
    },
    {
        'classifier': [LogisticRegression(random_state=RS)],
        'classifier__solver': ['saga'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__max_iter': [100, 500, 1000]
    },
    {
        'classifier': [RandomForestClassifier(random_state=RS)],
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_features': ['sqrt', 'log2'],
        'classifier__max_depth': [10, 20, None],
        'classifier__criterion': ['gini', 'entropy']
    },
    {
        'classifier': [SVC(random_state=RS)],
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto']
    }
]

pipe = Pipeline([('classifier', LogisticRegression(random_state=RS))])

r_search_multi_model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=params_extended,
    random_state=RS,
    cv=10,
    n_iter=50,
    scoring='accuracy',
    verbose=1
)

r_search_multi_model.fit(X_train, y_train)
best_overall_r_search_model = evaluate_tuned_model(r_search_multi_model, X_train, y_train, "RandomizedSearchCV (несколько моделей)")

Fitting 10 folds for each of 50 candidates, totalling 500 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Лучшие параметры для RandomizedSearchCV (несколько моделей): {'classifier__kernel': 'rbf', 'classifier__gamma': 'auto', 'classifier__C': 1, 'classifier': SVC(random_state=42)}
Лучшая оценка кросс-валидации для RandomizedSearchCV (несколько моделей): 0.8745
Лучшая общая модель: SVC(C=1, gamma='auto', random_state=42)

Метрики для лучшей модели RandomizedSearchCV (несколько моделей):
Accuracy (CV): 0.8745 (+/- 0.0256)
Recall (CV): 0.9151 (+/- 0.0280)
Precision (CV): 0.8656 (+/- 0.0382)
F1 (CV): 0.8888 (+/- 0.0206)


## Выводы:

**Метрики построенных моделей:**

*   **Базовая `LogisticRegression` (без тюнинга)**:
    * Accuracy (CV): 0.8506 (+/- 0.0582)
    * Recall (CV): 0.8695 (+/- 0.0992)
    * Precision (CV): 0.8667 (+/- 0.0637)
    * F1 (CV): 0.8639 (+/- 0.0571)

*   **`LogisticRegression` после `GridSearchCV`**:
    * Accuracy (CV): 0.8677 (+/- 0.0360)
    * Recall (CV): 0.8927 (+/- 0.0318)
    * Precision (CV): 0.8708 (+/- 0.0444)
    * F1 (CV): 0.8810 (+/- 0.0307)

*   **`LogisticRegression` после `RandomizedSearchCV` (только LogisticRegression)**:
    * Accuracy (CV): 0.8677 (+/- 0.0360)
    * Recall (CV): 0.8927 (+/- 0.0318)
    * Precision (CV): 0.8708 (+/- 0.0444)
    * F1 (CV): 0.8810 (+/- 0.0307)

*   **Лучшая модель после `RandomizedSearchCV` (несколько моделей: LogisticRegression, RandomForestClassifier, SVC)**:
    * Accuracy (CV): 0.8745 (+/- 0.0256)
    * Recall (CV): 0.9151 (+/- 0.0280)
    * Precision (CV): 0.8656 (+/- 0.0382)
    * F1 (CV): 0.8888 (+/- 0.0206)

**Базовая модель показала самые низкие метрики. После подбора гиперпараметров с помощью `GridSearchCV` и `RandomizedSearchCV` все метрики выросли. Метрика `precision` в этих методах показала самое высокое значение. Оба этих метода показали одинаковые значения метрик на разных солверах**:
  * `{'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}` для `GridSearchCV`
  * `{'solver': 'saga', 'penalty': 'l2', 'max_iter': 100, 'C': 1}` для `RandomSearchCV`

**`RandomizedSearchCV` для нескольких моделей показала самые высокие значения метрик `accuracy`, `recall`,`f1` для модели `SVC` и параметров `{'classifier__kernel': 'rbf', 'classifier__gamma': 'auto', 'classifier__C': 1, 'classifier': SVC(random_state=42)}`**

